# Telemetry Analysis

Analyse des données de télémétrie machines. Chargement du CSV, puis génération des distributions par machine et par métrique.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

df = pd.read_csv("artifacts/updated_dataset/telemetry.csv")
df = df[df["machine_id"] != "machine_id"]  # drop duplicate headers
print(f"Shape: {df.shape}")
df.head()

## Chargement des données

Import des librairies et lecture du fichier `telemetry.csv`. On supprime les éventuelles lignes d'en-tête dupliquées.

## Génération des distributions

Pour chaque machine (MACH-01 à MACH-15) et chaque métrique (temperature, pressure, voltage, rotation_mean, pieces), génère un histogramme avec courbe KDE et l'exporte au format PNG dans `artifacts/telemetry/machines/`.

In [ ]:
OUTPUT_DIR = Path("artifacts/telemetry/machines")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METRICS = {
    "temperature": "temperature_c",
    "pressure": "pressure_bar",
    "voltage": "voltage_mean_v",
    "rotation_mean": "rotation_mean_rpm",
    "pieces": "pieces_produced",
}

machines = sorted(df["machine_id"].unique())

for machine in machines:
    df_machine = df[df["machine_id"] == machine]
    for metric_name, col in METRICS.items():
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.histplot(df_machine[col].dropna().astype(float), kde=True, ax=ax, color="steelblue")
        ax.set_title(f"{machine} — {metric_name} distribution")
        ax.set_xlabel(col)
        ax.set_ylabel("Count")
        plt.tight_layout()
        filename = f"{machine}_{metric_name}_DISTRIBUTION.png"
        fig.savefig(OUTPUT_DIR / filename, dpi=100)
        plt.close(fig)

print(f"Done. {len(machines) * len(METRICS)} graphs saved to {OUTPUT_DIR}")

## Distribution globale par métrique

Distribution de chaque métrique sur l'ensemble des machines, sans distinction. Les graphiques sont exportés dans `artifacts/telemetry/machines/metrics/`.

In [ ]:
METRICS_OUTPUT_DIR = Path("artifacts/telemetry/metrics")
METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for metric_name, col in METRICS.items():
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(df[col].dropna().astype(float), kde=True, ax=ax, color="darkorange")
    ax.set_title(f"{metric_name} — global distribution (all machines)")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    plt.tight_layout()
    filename = f"{metric_name}_DISTRIBUTION.png"
    fig.savefig(METRICS_OUTPUT_DIR / filename, dpi=100)
    plt.close(fig)

print(f"Done. {len(METRICS)} graphs saved to {METRICS_OUTPUT_DIR}")